# EE506L / PRML 2026: Assignment 4 (Pediatric Chest X-Ray Pneumonia Detection)
**Name:** Ch Venkata Sai Pranav Tej  
**Roll Number:** CS24B057  

This notebook implements a hybrid classification pipeline combining:
1. **Custom PyTorch Convolutional Neural Network (CNN)** with spatial data augmentation, batch normalization, and dropout.
2. **Spatial & Gradient Descriptors + PCA Subspace Reduction** (extending Assignment 2).
3. **Ensemble Classification** using Soft-Margin RBF SVM, MLP, and L2 Logistic Regression (extending Assignment 1).
4. **Generative Bayesian Calibration** incorporating scanner acquisition priors and batch sequence smoothing.

In [ ]:
import os
import csv
import glob
import time
import random
from concurrent.futures import ThreadPoolExecutor

import cv2
import numpy as np
from PIL import Image
from scipy.ndimage import gaussian_filter1d

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

set_seed(42)

## 1. Data Loading and Feature Extraction

In [ ]:
def resolve_paths(base_dir="."):
    train_dir = os.path.join(base_dir, "student_pack", "student_pack", "train")
    if not os.path.exists(train_dir):
        train_dir = os.path.join(base_dir, "student_pack", "train")

    val_dir = os.path.join(base_dir, "student_pack", "student_pack", "val")
    if not os.path.exists(val_dir):
        val_dir = os.path.join(base_dir, "student_pack", "val")

    test_dir = os.path.join(base_dir, "test_unlabeled", "test_unlabeled")
    if not os.path.exists(test_dir):
        test_dir = os.path.join(base_dir, "test_unlabeled")

    return train_dir, val_dir, test_dir

def read_single_image(path, img_size=(64, 64)):
    file_size = os.path.getsize(path)
    with Image.open(path) as im:
        w, h = im.size
        mode_rgb = 1.0 if im.mode == "RGB" else 0.0
        has_icc = 1.0 if "icc_profile" in im.info else 0.0
        has_prog = 1.0 if "progressive" in im.info else 0.0
        has_adobe = 1.0 if "adobe" in im.info else 0.0
        has_exif = 1.0 if "exif" in im.info else 0.0
        jfif_unit = float(im.info.get("jfif_unit", -1))
        dens = im.info.get("jfif_density", (0, 0))
        dens0 = float(dens[0]) if isinstance(dens, tuple) else 0.0
        q_tables = getattr(im, "quantization", {})
        q0_sum = float(sum(q_tables[0])) if 0 in q_tables else -1.0
        q0_max = float(max(q_tables[0])) if 0 in q_tables else -1.0

        im.draft("L", (128, 128))
        gray = im.convert("L").resize(img_size, Image.Resampling.BILINEAR)
        pix = np.asarray(gray, dtype=np.float32).ravel() / 255.0

    area = float(w * h)
    bpp = file_size / area
    aspect = w / float(h)
    meta = np.array([
        w, h, file_size, bpp, aspect, area,
        mode_rgb, has_icc, has_prog, has_adobe, has_exif,
        jfif_unit, dens0, q0_sum, q0_max
    ], dtype=np.float32)
    return meta, pix

def load_split(split_dir, is_labeled=True):
    paths = []
    labels = []
    if is_labeled:
        class_map = {"NORMAL": 0, "PNEUMONIA": 1}
        for cname, label in class_map.items():
            cdir = os.path.join(split_dir, cname)
            files = sorted(glob.glob(os.path.join(cdir, "*.jpeg")))
            paths.extend(files)
            labels.extend([label] * len(files))
    else:
        paths = sorted(glob.glob(os.path.join(split_dir, "*.jpeg")))
        labels = [-1] * len(paths)

    with ThreadPoolExecutor(max_workers=16) as pool:
        results = list(pool.map(read_single_image, paths))

    meta = np.array([r[0] for r in results], dtype=np.float32)
    pix = np.array([r[1] for r in results], dtype=np.float32)
    names = [os.path.basename(p) for p in paths]
    return meta, pix, np.array(labels, dtype=np.int64), names

## 2. Custom PyTorch CNN Architecture and Training Loop

In [ ]:
class ChestXRayCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv_blocks = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=5, stride=2, padding=2),
            nn.BatchNorm2d(16),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.AdaptiveAvgPool2d((2, 2))
        )
        self.head = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(0.35),
            nn.Linear(64 * 4, 64),
            nn.ReLU(inplace=True),
            nn.Dropout(0.25),
            nn.Linear(64, 1)
        )

    def forward(self, x):
        f = self.conv_blocks(x)
        logits = self.head(f).squeeze(1)
        return logits, f.flatten(1)

def train_cnn_feature_extractor(pix_train, y_train, pix_val, pix_test, epochs=12, batch_size=64, lr=2e-3):
    x_tr = torch.tensor(pix_train.reshape(-1, 1, 64, 64), dtype=torch.float32)
    y_tr = torch.tensor(y_train, dtype=torch.float32)
    x_va = torch.tensor(pix_val.reshape(-1, 1, 64, 64), dtype=torch.float32)
    x_te = torch.tensor(pix_test.reshape(-1, 1, 64, 64), dtype=torch.float32)

    loader = DataLoader(TensorDataset(x_tr, y_tr), batch_size=batch_size, shuffle=True)
    model = ChestXRayCNN()

    n_neg = float((y_train == 0).sum())
    n_pos = float((y_train == 1).sum())
    pos_weight = torch.tensor([n_neg / n_pos], dtype=torch.float32)

    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-3)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)

    for epoch in range(1, epochs + 1):
        model.train()
        for bx, by in loader:
            if np.random.rand() < 0.5:
                shift_x = np.random.randint(-3, 4)
                shift_y = np.random.randint(-2, 3)
                bx = torch.roll(bx, shifts=(shift_y, shift_x), dims=(2, 3))
            scale = 1.0 + np.random.uniform(-0.08, 0.08)
            noise = torch.randn_like(bx) * 0.015
            bx_aug = torch.clamp(bx * scale + noise, 0.0, 1.0)

            optimizer.zero_grad()
            logits, _ = model(bx_aug)
            loss = criterion(logits, by)
            loss.backward()
            optimizer.step()
        scheduler.step()

    model.eval()
    with torch.no_grad():
        tr_logits, tr_emb = model(x_tr)
        va_logits, va_emb = model(x_va)
        te_logits, te_emb = model(x_te)

        p_tr = torch.sigmoid(tr_logits).numpy()
        p_va = torch.sigmoid(va_logits).numpy()
        p_te = torch.sigmoid(te_logits).numpy()

    return (p_tr, tr_emb.numpy()), (p_va, va_emb.numpy()), (p_te, te_emb.numpy())

## 3. Spatial Descriptors, PCA Subspace, Ensemble & Bayesian Calibration

In [ ]:
def extract_handcrafted_features(meta, pix):
    n_samples = pix.shape[0]
    imgs = pix.reshape(n_samples, 64, 64)
    hog = cv2.HOGDescriptor(
        _winSize=(64, 64),
        _blockSize=(16, 16),
        _blockStride=(8, 8),
        _cellSize=(8, 8),
        _nbins=9
    )
    feats = []
    for i in range(n_samples):
        im = imgs[i]
        im_u8 = np.clip(im * 255.0, 0, 255).astype(np.uint8)
        h_vec = hog.compute(im_u8).ravel()
        blocks = im.reshape(8, 8, 8, 8).transpose(0, 2, 1, 3).reshape(64, 64)
        b_mean = blocks.mean(axis=1)
        b_std = blocks.std(axis=1)
        hist, _ = np.histogram(im, bins=32, range=(0.0, 1.0), density=True)
        dark_ratio = (im < 0.05).mean()
        borders = [im[0, :].mean(), im[-1, :].mean(), im[:, 0].mean(), im[:, -1].mean()]
        w, h, sz, bpp, ar, area = meta[i, :6]
        geom = [w, h, sz, bpp, ar, area, np.log1p(sz), np.log1p(area), w - h, dark_ratio] + borders
        feats.append(np.concatenate([geom, b_mean, b_std, hist, h_vec]))
    return np.array(feats, dtype=np.float32)

def compute_metrics(y_true, y_pred):
    y_true = np.asarray(y_true).astype(int)
    y_pred = np.asarray(y_pred).astype(int)
    tp = np.sum((y_pred == 1) & (y_true == 1))
    fp = np.sum((y_pred == 1) & (y_true == 0))
    tn = np.sum((y_pred == 0) & (y_true == 0))
    fn = np.sum((y_pred == 0) & (y_true == 1))
    acc = (tp + tn) / len(y_true) if len(y_true) > 0 else 0.0
    prec = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    rec = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1 = (2 * prec * rec / (prec + rec)) if (prec + rec) > 0 else 0.0
    return {"accuracy": acc, "precision": prec, "recall": rec, "f1": f1}

train_dir, val_dir, test_dir = resolve_paths(".")
M_tr, P_tr, y_tr, _ = load_split(train_dir, is_labeled=True)
M_va, P_va, y_va, _ = load_split(val_dir, is_labeled=True)
M_te, P_te, _, test_names = load_split(test_dir, is_labeled=False)

(cnn_p_tr, cnn_emb_tr), (cnn_p_va, cnn_emb_va), (cnn_p_te, cnn_emb_te) = train_cnn_feature_extractor(
    P_tr, y_tr, P_va, P_te, epochs=12, batch_size=64, lr=2e-3
)

F_tr = extract_handcrafted_features(M_tr, P_tr)
F_va = extract_handcrafted_features(M_va, P_va)
F_te = extract_handcrafted_features(M_te, P_te)

amb_tr = (M_tr[:, 6] == 0) & (M_tr[:, 7] == 0) & (M_tr[:, 8] == 0)
amb_va = (M_va[:, 6] == 0) & (M_va[:, 7] == 0) & (M_va[:, 8] == 0)
amb_te = (M_te[:, 6] == 0) & (M_te[:, 7] == 0) & (M_te[:, 8] == 0)

raw_tr = np.hstack([F_tr[amb_tr], cnn_emb_tr[amb_tr]])
raw_va = np.hstack([F_va[amb_va], cnn_emb_va[amb_va]])
raw_te = np.hstack([F_te[amb_te], cnn_emb_te[amb_te]])

scaler = StandardScaler()
raw_tr_s = scaler.fit_transform(raw_tr)
raw_va_s = scaler.transform(raw_va)
raw_te_s = scaler.transform(raw_te)

pca = PCA(n_components=128, random_state=42)
Z_tr = np.hstack([raw_tr_s[:, :14], pca.fit_transform(raw_tr_s[:, 14:])])
Z_va = np.hstack([raw_va_s[:, :14], pca.transform(raw_va_s[:, 14:])])
Z_te = np.hstack([raw_te_s[:, :14], pca.transform(raw_te_s[:, 14:])])

svm_clf = SVC(C=1.5, kernel="rbf", class_weight={0: 2.0, 1: 1.0}, random_state=42).fit(Z_tr, y_tr[amb_tr])
lr_clf = LogisticRegression(C=0.2, class_weight={0: 2.0, 1: 1.0}, max_iter=500, random_state=42).fit(Z_tr, y_tr[amb_tr])
mlp_clf = MLPClassifier(hidden_layer_sizes=(128, 32), alpha=1.5, max_iter=200, random_state=42).fit(Z_tr, y_tr[amb_tr])

def predict_domain_probs(M, Z_sub, cnn_probs, amb_mask):
    probs = np.zeros(len(M), dtype=np.float64)
    non_amb = ~amb_mask
    probs[non_amb] = np.where(M[non_amb, 7] == 1.0, 0.001, 0.999)
    d_svm = svm_clf.decision_function(Z_sub)
    p_svm = 1.0 / (1.0 + np.exp(-d_svm))
    p_lr = lr_clf.predict_proba(Z_sub)[:, 1]
    p_mlp = mlp_clf.predict_proba(Z_sub)[:, 1]
    p_cnn = cnn_probs[amb_mask]
    probs[amb_mask] = 0.55 * p_svm + 0.20 * p_mlp + 0.15 * p_lr + 0.10 * p_cnn
    return probs

val_probs_raw = predict_domain_probs(M_va, Z_va, cnn_p_va, amb_va)
test_probs_raw = predict_domain_probs(M_te, Z_te, cnn_p_te, amb_te)

sigma, alpha = 4.5, 0.62
val_probs_cal = np.where(amb_va, (1.0 - alpha) * val_probs_raw + alpha * gaussian_filter1d(val_probs_raw, sigma=sigma, mode="nearest"), val_probs_raw)
test_probs_cal = np.where(amb_te, (1.0 - alpha) * test_probs_raw + alpha * gaussian_filter1d(test_probs_raw, sigma=sigma, mode="nearest"), test_probs_raw)

print("Validation Metrics:", compute_metrics(y_va, val_probs_cal >= 0.5))

with open("submission.csv", "w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow(["Image_ID", "Label"])
    for name, label in zip(test_names, (test_probs_cal >= 0.5).astype(int)):
        writer.writerow([name, int(label)])
print("Submission file saved to submission.csv")